In [ ]:
!pip install transformers
import pandas as pd
import numpy as np
import torch
import random

from transformers import AutoTokenizer
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from sklearn.metrics import f1_score
from transformers import Trainer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from transformers import set_seed

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


#Check the label distribution in the training set
train = pd.read_json("/kaggle/input/datasets/zelenezhong/dataset-task-1/train.jsonl", lines=True)

print(train["tags"].value_counts())

See that we have three types of tags here, "phrase", "passage", or "multi". Conforming to the descriptions in Task 1.

In [ ]:
#See the first few rows for the training set

train.head()

In [ ]:
#Also, see the column names of the train dataset
train.columns

Based on the dataset description, 14 variables are available. The target variable is tags, which represents three spoiler categories: phrase, passage, and multi.

In [ ]:
#Read the test dataset as well as the validation set
test = pd.read_json("/kaggle/input/datasets/zelenezhong/dataset-task-1/test.jsonl", lines=True)

val = pd.read_json("/kaggle/input/datasets/zelenezhong/dataset-task-1/val.jsonl", lines=True)

#See what variables are available in the test dataset
test.columns

For the input variables, we mainly focus on postText, targetTitle, and targetParagraphs. The post text captures the linguistic characteristics of clickbait expressions, while the webpage title and paragraphs provide contextual information about the underlying content and spoiler type. These textual features are then transformed into model inputs through tokenization.

In [ ]:
#Convert the three predictors into the input of the model.
def create_text(df):

    df["paragraph_text"] = df["targetParagraphs"].apply(
        lambda x: " ".join(x)
    )

    df["post_text"] = df["postText"].apply(
    lambda x: " ".join(x)
)

    df["text"] = (
        df["post_text"]
        + " "
        + df["targetTitle"]
        + " "
        + df["paragraph_text"]
    )

    return df

train_df = create_text(train)
val_df = create_text(val)
test_df = create_text(test)

train_df["text"].head()

In [ ]:
#Then, we would like to deal with the target variable, mapping them to 0, 1, and 2
label_mapping = {
    "phrase":0,
    "passage":1,
    "multi":2
}

train_df["label"] = train_df["tags"].apply(
    lambda x: label_mapping[x[0]]
)

val_df["label"] = val_df["tags"].apply(
    lambda x: label_mapping[x[0]]
)

Based on the distribution of tags, the three spoiler categories are relatively imbalanced. We analyze the text length distribution and examine the token length after applying the tokenizer.

In [ ]:
train_df["text"].apply(
    lambda x: len(x.split())
).describe()

In [ ]:
#We use the Roberta model to check tokenizer, as the strength of it is understanding and classification
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df["text"].apply(
    lambda x: len(tokenizer.encode(x))
).describe()

The token length analysis shows that some input sequences exceed the maximum sequence length of 512 tokens. Therefore, the original text cannot be directly used as input without further preprocessing.

To address this issue, we apply sequence length handling strategies before model training.

In [ ]:
#We can reduce the number of paragraphs, redefine the text input.
#use the first 3 in this case.
#change the formatting of the input
def create_text_new(df):

    df["paragraph_text"] = df["targetParagraphs"].apply(
    lambda x: " ".join(x[:3])
)

    df["post_text"] = df["postText"].apply(
    lambda x: " ".join(x)
)

    df["text"] = (
        "Post: "
        + df["post_text"]
        + " Title: "
        + df["targetTitle"]
        + " Article: "
        + df["paragraph_text"]
    )

    return df

train_df_new = create_text_new(train)
val_df_new = create_text_new(val)
test_df_new = create_text_new(test)

lengths = train_df_new["text"].apply(
    lambda x: len(tokenizer.encode(x))
)

lengths.describe()

Based on the results, this version of text preprocessing is more suitable for Roberta compared with the previous approach. However, we still need to evaluate the proportion of input sequences that exceed the maximum length limit of 512 tokens.

In [ ]:
#Count how many are over 512.
(lengths > 512).sum()

In [ ]:
#Count the percentage of them over 512.
(lengths > 512).mean()

This means that 0.6% of the samples exceed 512 tokens. Approximately 99.4% were within the limit. This means that the vast majority of texts will not be truncated and only a small number of long texts will lose the latter part.

Then, the tokenizer applies truncation to longer sequences and padding to shorter sequences, ensuring that all inputs have a fixed length compatible with the model.

In [ ]:
train_encodings = tokenizer(
    train_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

val_encodings = tokenizer(
    val_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

test_encodings = tokenizer(
    test_df_new["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=512
)

In [ ]:
#Create Dataset
class ClickbaitDataset(Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])

        return item

After tokenization, the data are organized into training, validation, and test datasets. The training and validation datasets contain input features and labels, while the test dataset only contains input features because the true labels are not provided for evaluation.

In [ ]:
#Notice that no 'label' for test dataset
train_dataset = ClickbaitDataset(
    train_encodings,
    train_df_new["label"].tolist()
)

val_dataset = ClickbaitDataset(
    val_encodings,
    val_df_new["label"].tolist()
)

test_dataset = ClickbaitDataset(
    test_encodings
)

Then, load the classification model

In [ ]:
num_labels = 3 #As we have three classes

#Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

After that, the training parameters are configured, including the learning rate, batch size, number of training epochs, and weight decay. These parameters control the optimization process and affect the model's performance during fine-tuning.

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_input_3_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="weighted_f1",

    greater_is_better=True
)

We fine-tuned a pretrained model for spoiler type classification. The model was trained for 3 epochs with a learning rate of 2e-5 and batch size of 16. The maximum input sequence length was set to 512 tokens. Validation performance was monitored after each epoch using weighted F1-score, and the best-performing checkpoint was selected.

Since the competition evaluates performance using F1-score, weighted F1-score was used to monitor validation performance after each epoch, and the best-performing checkpoint was selected based on this metric.

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "weighted_f1": f1
    }

Then we set up the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

Then, the configured Trainer is applied to the training dataset to fine-tune the Roberta model.

In [ ]:
trainer.train()

The fine-tuned model is evaluated on the validation dataset using the Trainer's evaluation function. The evaluation results include validation loss and weighted F1-score, which are used to assess the model's classification performance.

In [ ]:
#See the evaluate score
trainer.evaluate()

To further analyze the classification performance, predictions were generated on the validation dataset. The model outputs logits for each class, and the class with the highest probability was selected as the predicted label using the argmax function. The predicted labels were then compared with the true validation labels for performance evaluation.

In [ ]:
val_output = trainer.predict(val_dataset)

val_logits = val_output.predictions

val_pred = val_logits.argmax(axis=1)

val_labels = val_df_new["label"].values

In [ ]:
print(
    classification_report(
        val_labels,
        val_pred,
        target_names=[
            "phrase",
            "passage",
            "multi"
        ]
    )
)

Then we can produce a confusion matrix for the validation set.

In [ ]:
cm = confusion_matrix(
    val_labels,
    val_pred
)

pd.DataFrame(
    cm,
    index=["phrase","passage","multi"],
    columns=["phrase","passage","multi"]
)

After the validation process, the model is applied to the test dataset to generate final predictions. Since the test labels are not provided, the model outputs are used directly to determine the predicted spoiler types for each test sample.

In [ ]:
test_output = trainer.predict(test_dataset)

test_pred = test_output.predictions.argmax(axis=1)

#Use the label we have for mapping
id_to_label = {
    0:"phrase",
    1:"passage",
    2:"multi"
}

test_types = [
    id_to_label[x]
    for x in test_pred
]

In [ ]:
#Then we have the submission
submission_task1 = pd.DataFrame({
    "id": test_df["id"],
    "spoilerType": test_types
})

#Check the head of the submission
submission_task1.head()

Finally, the predicted spoiler types are saved as a CSV file. This file contains the predicted labels for all test samples and is submitted for evaluation.

In [ ]:
submission_task1.to_csv(
    "prediction_task1.csv",
    index=False
)